# Animagine XL 4.0 + LPW • Gradio trên Colab
Chạy các ô theo thứ tự. Bật **Runtime → Change runtime type → GPU** trước.

**Quan trọng:** Colab miễn phí không cho phép dùng chủ yếu qua web UI thay notebook và có thể ngắt phiên. Dùng gói trả phí còn compute units hoặc máy GPU riêng cho giao diện này. Xem [FAQ Colab](https://research.google.com/colaboratory/faq.html#disallowed-activities).

Link `gradio.live` là link tạm, không phải hosting GPU; Colab dừng thì link ngừng hoạt động. Không có cơ chế giữ phiên/lách giới hạn. Tải ảnh về trước khi kết thúc phiên.

LPW chia prompt thành đoạn; nhận nhiều token không đảm bảo hiểu mọi chi tiết. Giao diện giới hạn 300 token mỗi encoder cho mỗi prompt để hạn chế VRAM, không âm thầm cắt prompt.


## 1. Cài thư viện
Dùng runtime mới. Nếu Colab yêu cầu restart, restart rồi chạy từ ô 2. Giữ nguyên PyTorch do Colab cung cấp.

In [ ]:
%pip -q install "diffusers==0.35.1" "transformers==4.56.2" "accelerate==1.10.1" "huggingface-hub==0.35.3" "safetensors==0.6.2" "gradio==5.49.1"

## 2. Tải model (vài GB)
FP16 và CPU offload giúp giảm VRAM, đổi lại chậm hơn. Community pipeline là mã tải từ Diffusers, ghim cùng phiên bản thư viện.

In [ ]:
import torch, importlib, inspect
from diffusers import DiffusionPipeline, EulerAncestralDiscreteScheduler
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU rồi chạy lại.")
print("GPU:", torch.cuda.get_device_name(0))
if "pipe" not in globals():
    pipe = DiffusionPipeline.from_pretrained(
        "cagliostrolab/animagine-xl-4.0",
        custom_pipeline="lpw_stable_diffusion_xl",
        custom_revision="v0.35.1",
        torch_dtype=torch.float16, use_safetensors=True, add_watermarker=False,
    )
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_tiling()
    pipe.enable_vae_slicing()
lpw = importlib.import_module(type(pipe).__module__)
assert hasattr(lpw, "get_prompts_tokens_with_weights"), "Chưa tải đúng pipeline LPW."
print("Sẵn sàng:", type(pipe).__name__)


## 3. Prompt mẫu
Có thể sửa trực tiếp trên giao diện. `(blue eyes:1.2)` tăng trọng số; tránh quá nhiều tag mâu thuẫn.

In [ ]:
prompt = '''
1girl, solo, original anime character, long silver hair, blue eyes,
small black ribbon, gentle smile, looking at viewer,
wearing a navy and white school uniform, pleated skirt,
standing on a quiet train platform at dusk, holding a transparent umbrella,
gentle rain falling, wet pavement reflecting station lights,
a train approaching in the far background, distant city lights,
wind moving her hair and skirt, full body, three-quarter view,
character placed slightly to the right, cinematic composition,
soft blue and violet lighting, warm station lamps, atmospheric perspective,
detailed background, anime illustration, clean line art, expressive eyes,
delicate shading, masterpiece, high score, great score, absurdres
'''

negative_prompt = '''
low quality, worst quality, blurry, bad anatomy, bad hands, extra fingers,
missing fingers, extra limbs, malformed limbs, duplicate, text, watermark,
signature, cropped, jpeg artifacts
'''

print('Số ký tự prompt:', len(prompt))

## 4. Tạo ảnh • Sửa vùng • Upscale AI
Tác vụ GPU dùng chung hàng đợi. Upscale AI dùng Real-ESRGAN Anime6B (x4), giữ Lanczos làm lựa chọn CPU. Không cài BasicSR/torchvision để tránh lỗi tương thích với PyTorch Colab.


In [ ]:
import gradio as gr
import gc, secrets, time, json, math
from pathlib import Path
from uuid import uuid4

OUTPUT_DIR = Path("/content/animagine_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SIZES = {"Dọc · 832 × 1216": (832,1216), "Vuông · 1024 × 1024": (1024,1024),
         "Ngang · 1216 × 832": (1216,832), "Nhẹ · 768 × 1024": (768,1024)}

def counts(p, n):
    if len(p) > 6000 or len(n) > 6000:
        raise gr.Error("Mỗi prompt tối đa 6000 ký tự; hãy rút gọn.")
    return [[len(lpw.get_prompts_tokens_with_weights(t, text)[0])
             for t in (pipe.tokenizer, pipe.tokenizer_2)] for text in (p,n)]

def token_report(p,n):
    c = counts(p,n)
    return f"Prompt: {c[0]} • Negative: {c[1]} token [CLIP 1, CLIP 2]. Giới hạn: 300 mỗi encoder."

def generate(p,n,size,steps,cfg,seed,progress=gr.Progress()):
    if not p.strip(): raise gr.Error("Vui lòng nhập prompt.")
    c = counts(p,n)
    if max(c[0]+c[1]) > 300:
        raise gr.Error("Prompt vượt 300 token. Hãy rút gọn; ứng dụng không tự cắt nội dung.")
    # Pipeline cộng đồng ghép hai encoder: cần cùng số token trước khi ghép.
    if c[0][0] != c[0][1] or c[1][0] != c[1][1]:
        raise gr.Error("Hai tokenizer cho độ dài khác nhau. Hãy thử tag tiếng Anh đơn giản hơn.")
    if size not in SIZES: raise gr.Error("Kích thước không hợp lệ.")
    if not 10 <= steps <= 40 or not 1 <= cfg <= 10: raise gr.Error("Steps/CFG ngoài giới hạn.")
    if seed != int(seed) or not -1 <= seed <= 4294967295: raise gr.Error("Seed phải là số nguyên từ -1 đến 4294967295.")
    seed = secrets.randbelow(4294967296) if int(seed) == -1 else int(seed)
    width,height = SIZES[size]
    started = time.monotonic()
    progress(0, desc="Mã hóa prompt dài…")
    def on_step(_pipe, i, t, kwargs):
        progress((i+1)/int(steps), desc=f"Đang vẽ: {i+1}/{int(steps)}")
        return kwargs
    try:
        with torch.inference_mode():
            image = pipe(prompt=p, negative_prompt=n, width=width, height=height,
                         num_inference_steps=int(steps), guidance_scale=float(cfg),
                         generator=torch.Generator(device="cpu").manual_seed(seed),
                         callback_on_step_end=on_step).images[0]
    except torch.cuda.OutOfMemoryError:
        gc.collect()
        torch.cuda.empty_cache()
        raise gr.Error("Hết VRAM. Chọn 768 × 1024 và rút gọn prompt. Nếu vẫn lỗi, khởi động lại runtime.") from None
    stem = OUTPUT_DIR / f"anime_{seed}_{uuid4().hex[:12]}"
    png, meta = str(stem)+".png", str(stem)+".json"
    image.save(png)
    data = dict(model="cagliostrolab/animagine-xl-4.0", prompt=p, negative_prompt=n,
                seed=seed, width=width, height=height, steps=int(steps), cfg=float(cfg),
                sampler="Euler a", tokens=c, diffusers="0.35.1")
    Path(meta).write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding="utf-8")
    return png, png, meta, f"Hoàn tất • seed {seed} • {time.monotonic()-started:.1f}s\n" + token_report(p,n)

# CSV không có header: tag, category_id, post_count, aliases.
import csv, urllib.request
TAG_URL = "https://raw.githubusercontent.com/manhlee1196-boop/anime2/b65bea25a2dabf46859b70db3f78258b68de47a4/danbooru_e621_merged_2026-10-01_pt20-ia-dd-ed-spc.csv"
TAG_PATH = Path('/content/anime_tags.csv')
TAG_ROWS = []
TAG_NAMES = set()

def read_tags(path):
    rows = []
    with open(path, encoding='utf-8-sig', newline='') as f:
        for line, row in enumerate(csv.reader(f), 1):
            if len(row) != 4:
                raise ValueError(f'Dòng {line}: cần 4 cột.')
            name, category, count, aliases = row
            if not name.strip() or int(count) < 0:
                raise ValueError(f'Dòng {line}: tên/count không hợp lệ.')
            rows.append((name, int(category), int(count), aliases.lower()))
    return sorted(rows, key=lambda r: -r[2])

def load_tags(upload=None):
    global TAG_ROWS, TAG_NAMES
    try:
        if upload:
            path = Path(upload)
        else:
            if not TAG_PATH.exists():
                temporary = TAG_PATH.with_suffix('.tmp')
                with urllib.request.urlopen(TAG_URL, timeout=60) as response:
                    with temporary.open('wb') as f:
                        total = 0
                        while chunk := response.read(1024*1024):
                            total += len(chunk)
                            if total > 30*1024*1024: raise ValueError('CSV vượt 30 MB.')
                            f.write(chunk)
                temporary.replace(TAG_PATH)
            path = TAG_PATH
        if path.stat().st_size > 30*1024*1024: raise ValueError('CSV vượt 30 MB.')
        rows = read_tags(path)
    except Exception as exc:
        raise gr.Error('Không nạp được CSV. Kiểm tra file hoặc tải CSV từ GitHub rồi upload. ' + str(exc)) from None
    TAG_ROWS, TAG_NAMES = rows, {r[0] for r in rows}
    categories = ['Tất cả'] + [str(c) for c in sorted({r[1] for r in rows})]
    return f'Đã nạp {len(rows):,} thẻ.', gr.update(choices=categories, value='Tất cả')

def search_tags(query, category):
    if not TAG_ROWS: raise gr.Error('Bấm Nạp CSV trước.')
    q = query.strip().lower().replace(' ', '_')
    if len(q) < 2: return gr.update(choices=[], value=[])
    matches = []
    for name, cat, count, aliases in TAG_ROWS:
        if category != 'Tất cả' and str(cat) != category: continue
        lowered = name.lower()
        if q in lowered or q in aliases:
            rank = 0 if q == lowered else 1 if lowered.startswith(q) else 2 if q in lowered else 3
            matches.append((rank, -count, name, cat, count))
    matches.sort()
    choices = [(f'{name} · loại {cat} · {count:,}', name)
               for _, _, name, cat, count in matches[:80]]
    return gr.update(choices=choices, value=[])

def append_tags(text, selected, spaces):
    # Escape ký tự trọng số LPW trong tên thật, ví dụ character_(series).
    existing = {part.strip() for part in text.split(',')}
    added = []
    for tag in selected or []:
        if tag not in TAG_NAMES: continue
        tag = tag.replace('_', ' ') if spaces else tag
        tag = tag.replace('\\', '\\\\')
        for ch in '()[]': tag = tag.replace(ch, '\\' + ch)
        if tag not in existing:
            added.append(tag)
            existing.add(tag)
    return ', '.join(filter(None, [text.strip().rstrip(','), *added]))

from PIL import Image, ImageFilter, ImageOps
import numpy as np

MAX_IMAGE_PIXELS = 16_000_000

def open_image(value):
    if value is None: raise gr.Error("Chưa có ảnh. Tải ảnh lên hoặc chuyển ảnh từ tab Tạo ảnh.")
    if isinstance(value, Image.Image): image = value.copy()
    elif isinstance(value, np.ndarray): image = Image.fromarray(value)
    else:
        with Image.open(value) as src:
            if src.width * src.height > MAX_IMAGE_PIXELS: raise gr.Error("Ảnh quá lớn (tối đa 16 megapixel).")
            image = ImageOps.exif_transpose(src).copy()
    if image.width * image.height > MAX_IMAGE_PIXELS: raise gr.Error("Ảnh quá lớn (tối đa 16 megapixel).")
    return image.convert('RGB')

def editor_value(value):
    image = open_image(value)
    return {"background": image, "layers": [], "composite": image}

def extract_mask(value):
    if not isinstance(value, dict) or value.get('background') is None:
        raise gr.Error("Hãy tải ảnh vào khung và tô vùng cần sửa.")
    image = open_image(value['background'])
    mask = Image.new('L', image.size, 0)
    # Chỉ lấy alpha của lớp cọ, KHÔNG lấy composite (có ảnh nền).
    for layer in value.get('layers') or []:
        layer = Image.fromarray(layer) if isinstance(layer, np.ndarray) else layer
        if not isinstance(layer, Image.Image) or layer.mode != 'RGBA':
            raise gr.Error("Mask cần lớp cọ RGBA. Hãy xóa mask rồi tô lại bằng cọ.")
        alpha = layer.getchannel('A').resize(image.size, Image.Resampling.NEAREST)
        mask = Image.fromarray(np.maximum(np.asarray(mask), np.asarray(alpha)))
    mask = mask.point(lambda x: 255 if x > 0 else 0)
    if not mask.getbbox(): raise gr.Error("Chưa tô vùng sửa. Dùng cọ tô lên tay, chân hoặc mắt.")
    return image, mask

def repair_box(image, mask, crop, padding):
    if not crop: return (0, 0, image.width, image.height)
    x0,y0,x1,y1 = mask.getbbox()
    pad = int(padding)
    x0,y0,x1,y1 = max(0,x0-pad),max(0,y0-pad),min(image.width,x1+pad),min(image.height,y1+pad)
    # Bổ sung ngữ cảnh, tránh vùng sửa quá nhỏ/hẹp.
    side = max(x1-x0,y1-y0,256)
    cx,cy = (x0+x1)/2,(y0+y1)/2
    w,h = min(side,image.width),min(side,image.height)
    left = min(max(0,int(cx-w/2)),image.width-w)
    top = min(max(0,int(cy-h/2)),image.height-h)
    return left,top,left+w,top+h

def inference_size(size):
    w,h = size
    scale = 1024/max(w,h)
    return max(64,round(w*scale/64)*64),max(64,round(h*scale/64)*64)

def merge_repair(original, repaired, mask, box, feather):
    # Ghép ảnh ngoài pipeline: ngoài vùng feather giữ pixel ảnh gốc.
    patch = original.copy()
    patch.paste(repaired.resize((box[2]-box[0],box[3]-box[1]),Image.Resampling.LANCZOS),box[:2])
    soft = mask.filter(ImageFilter.GaussianBlur(float(feather))) if feather else mask
    return Image.composite(patch, original, soft)

def validated_seed(seed):
    try:
        if not math.isfinite(seed) or seed != int(seed) or not -1 <= seed <= 4294967295: raise ValueError()
    except (TypeError, ValueError, OverflowError):
        raise gr.Error("Seed phải là số nguyên từ -1 đến 4294967295.") from None
    return secrets.randbelow(4294967296) if int(seed)==-1 else int(seed)

def validate_repair(p,n,strength,steps,cfg,padding,feather):
    if not p.strip(): raise gr.Error("Hãy nhập prompt mô tả vùng sửa.")
    c = counts(p,n)
    if max(c[0]+c[1]) > 300: raise gr.Error("Prompt vượt 300 token; hãy rút gọn.")
    if c[0][0]!=c[0][1] or c[1][0]!=c[1][1]: raise gr.Error("Hai tokenizer khác độ dài. Thử tag tiếng Anh đơn giản hơn.")
    if not (0.15<=strength<=1 and 10<=steps<=40 and 1<=cfg<=10 and 0<=padding<=256 and 0<=feather<=32):
        raise gr.Error("Thông số sửa ngoài giới hạn.")
    return c

def save_edit(image,kind,data):
    stem = OUTPUT_DIR / f"{kind}_{uuid4().hex[:12]}"
    png,meta = str(stem)+'.png',str(stem)+'.json'
    image.save(png)
    Path(meta).write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
    return png,meta

def repair(value,p,n,strength,steps,cfg,seed,crop,padding,feather,progress=gr.Progress()):
    image,mask = extract_mask(value)
    c = validate_repair(p,n,strength,steps,cfg,padding,feather)
    seed = validated_seed(seed)
    box = repair_box(image,mask,crop,padding)
    region = image.crop(box)
    size = inference_size(region.size)
    input_image = region.resize(size,Image.Resampling.LANCZOS)
    input_mask = mask.crop(box).resize(size,Image.Resampling.NEAREST)
    started = time.monotonic()
    progress(0,desc="Đang sửa vùng…")
    try:
        with torch.inference_mode():
            result = pipe(prompt=p,negative_prompt=n,image=input_image,mask_image=input_mask,
                          width=size[0],height=size[1],strength=float(strength),
                          num_inference_steps=int(steps),guidance_scale=float(cfg),
                          generator=torch.Generator(device='cpu').manual_seed(seed)).images[0]
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        raise gr.Error("Hết VRAM. Rút gọn prompt, bật crop vùng sửa; nếu còn lỗi hãy restart runtime.") from None
    merged = merge_repair(image,result,mask,box,feather)
    png,meta = save_edit(merged,'repair',dict(model=MODEL_ID,mode='inpainting',seed=seed,
        prompt=p,negative_prompt=n,strength=float(strength),steps=int(steps),cfg=float(cfg),
        crop=bool(crop),box=box,inference_size=size,feather=float(feather),tokens=c))
    before,_ = save_edit(image,'before_repair',dict(mode='original_before_repair'))
    return image,png,png,meta,before,f"Đã sửa • seed {seed} • {time.monotonic()-started:.1f}s. Ngoài mask/feather giữ ảnh gốc."

def use_repaired(path):
    if not path: raise gr.Error("Chưa có kết quả sửa.")
    return editor_value(path)

def undo_repair(before):
    if not before: raise gr.Error("Chưa có bước sửa để hoàn tác.")
    image = open_image(before)
    return editor_value(image), image, None, None, None, None, "Đã hoàn tác bước sửa gần nhất."

# Adapted RRDB x4 architecture from BasicSR (2018–2022 BasicSR Authors), Apache-2.0.
# Changes: x4-only, six blocks, no registry/training initialization or torchvision dependency.
# Full license is included in the notebook's last markdown cell.
def build_anime6b():
    import torch
    from torch import nn
    from torch.nn import functional as F

    class ResidualDenseBlock(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(64,32,3,1,1)
            self.conv2 = nn.Conv2d(96,32,3,1,1)
            self.conv3 = nn.Conv2d(128,32,3,1,1)
            self.conv4 = nn.Conv2d(160,32,3,1,1)
            self.conv5 = nn.Conv2d(192,64,3,1,1)
            self.lrelu = nn.LeakyReLU(0.2,inplace=True)
        def forward(self,x):
            x1 = self.lrelu(self.conv1(x))
            x2 = self.lrelu(self.conv2(torch.cat((x,x1),1)))
            x3 = self.lrelu(self.conv3(torch.cat((x,x1,x2),1)))
            x4 = self.lrelu(self.conv4(torch.cat((x,x1,x2,x3),1)))
            return self.conv5(torch.cat((x,x1,x2,x3,x4),1))*0.2+x

    class RRDB(nn.Module):
        def __init__(self):
            super().__init__()
            self.rdb1,self.rdb2,self.rdb3 = ResidualDenseBlock(),ResidualDenseBlock(),ResidualDenseBlock()
        def forward(self,x):
            return self.rdb3(self.rdb2(self.rdb1(x)))*0.2+x

    class Anime6B(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv_first = nn.Conv2d(3,64,3,1,1)
            self.body = nn.Sequential(*[RRDB() for _ in range(6)])
            self.conv_body = nn.Conv2d(64,64,3,1,1)
            self.conv_up1 = nn.Conv2d(64,64,3,1,1)
            self.conv_up2 = nn.Conv2d(64,64,3,1,1)
            self.conv_hr = nn.Conv2d(64,64,3,1,1)
            self.conv_last = nn.Conv2d(64,3,3,1,1)
            self.lrelu = nn.LeakyReLU(0.2,inplace=True)
        def forward(self,x):
            feat = self.conv_first(x)
            feat = feat + self.conv_body(self.body(feat))
            feat = self.lrelu(self.conv_up1(F.interpolate(feat,scale_factor=2,mode='nearest')))
            feat = self.lrelu(self.conv_up2(F.interpolate(feat,scale_factor=2,mode='nearest')))
            return self.conv_last(self.lrelu(self.conv_hr(feat)))
    return Anime6B()

AI_MODEL_URL = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth'
AI_WEIGHTS = Path('/content/anime_models/RealESRGAN_x4plus_anime_6B.pth')
AI_MODEL = None
AI_MODE = 'Real-ESRGAN Anime6B (AI)'
LIGHT_MODE = 'Lanczos (nhẹ, không AI)'

def download_ai_weights(path=AI_WEIGHTS):
    import urllib.request
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and path.stat().st_size == 17938799: return path
    tmp = path.with_suffix('.download')
    try:
        with urllib.request.urlopen(AI_MODEL_URL,timeout=60) as response, tmp.open('wb') as f:
            total = 0
            while chunk := response.read(1024*1024):
                total += len(chunk)
                if total > 32*1024*1024: raise ValueError('File model vượt giới hạn.')
                f.write(chunk)
        if tmp.stat().st_size != 17938799:
            raise ValueError('File tải không đúng kích thước checkpoint chính thức.')
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)
    return path

def load_ai_model(progress):
    global AI_MODEL
    import torch
    if AI_MODEL is None:
        progress(0,desc='Tải Real-ESRGAN Anime6B lần đầu (~18 MB)…')
        try:
            path = download_ai_weights()
            # Không cho phép pickle thực thi tùy ý; chỉ nạp weights từ URL chính thức.
            checkpoint = torch.load(path,map_location='cpu',weights_only=True)
            state = checkpoint.get('params_ema',checkpoint.get('params'))
            if state is None: raise ValueError('Checkpoint không có params/params_ema.')
            model = build_anime6b()
            model.load_state_dict(state,strict=True)
            AI_MODEL = model.eval().requires_grad_(False)
        except Exception as exc:
            raise gr.Error('Không nạp được Anime6B. Kiểm tra Internet/ô cài thư viện, hoặc chọn Lanczos. ' + str(exc)) from None
    return AI_MODEL

def tiled_anime4x(image, model, tile, progress):
    import torch
    import numpy as np
    from PIL import Image
    if tile not in (64,128,256): raise ValueError('Tile phải là 64, 128 hoặc 256.')
    w,h = image.size
    # Global reflect padding giống nguyên lý Real-ESRGAN, dùng CPU cho ảnh/output lớn.
    pad = min(10,w-1,h-1)
    array = np.asarray(image.convert('RGB'))
    padded = np.pad(array,((pad,pad),(pad,pad),(0,0)),mode='reflect') if pad else array
    canvas = Image.new('RGB',(w*4,h*4))
    parameter = next(model.parameters())
    total = math.ceil(w/tile)*math.ceil(h/tile)
    done = 0
    for y in range(0,h,tile):
        for x in range(0,w,tile):
            x1,y1 = min(x+tile,w),min(y+tile,h)
            # 64 pixel ngữ cảnh quanh mỗi tile giúp giảm đường nối.
            left,top = max(0,x+pad-64),max(0,y+pad-64)
            right,bottom = min(w+2*pad,x1+pad+64),min(h+2*pad,y1+pad+64)
            block = np.ascontiguousarray(padded[top:bottom,left:right].transpose(2,0,1))
            tensor = torch.from_numpy(block).unsqueeze(0).to(device=parameter.device,dtype=parameter.dtype)/255
            with torch.inference_mode():
                out = model(tensor)
            out = out[0,:, (y+pad-top)*4:(y1+pad-top)*4, (x+pad-left)*4:(x1+pad-left)*4]
            rgb = out.float().clamp(0,1).mul(255).round().byte().permute(1,2,0).cpu().numpy()
            canvas.paste(Image.fromarray(rgb),(x*4,y*4))
            del tensor,out,rgb
            done += 1
            progress(done/total,desc=f'AI upscale: ô {done}/{total}')
    return canvas

def run_ai_upscale(image,tile,progress):
    import torch
    if not torch.cuda.is_available(): raise gr.Error('AI upscale cần GPU. Bật GPU trong Colab hoặc chọn Lanczos.')
    # Giải phóng model diffusion khỏi GPU; giữ nguyên accelerate hooks để tạo/sửa sau đó.
    for name in ('text_encoder','text_encoder_2','unet','vae'):
        component = getattr(pipe,name,None)
        if component is not None: component.to('cpu')
    gc.collect(); torch.cuda.empty_cache()
    model = load_ai_model(progress)
    try:
        model.to(device='cuda',dtype=torch.float16)
        return tiled_anime4x(image,model,int(tile),progress)
    except torch.cuda.OutOfMemoryError:
        raise gr.Error('Hết VRAM. Chọn tile 64, ảnh nhỏ hơn hoặc dùng Lanczos. Nếu còn lỗi hãy restart runtime.') from None
    finally:
        # Không để upscaler chiếm VRAM khi quay lại Tạo ảnh/Sửa vùng.
        model.to(device='cpu',dtype=torch.float32)
        gc.collect(); torch.cuda.empty_cache()


def upscale(value,factor,sharpen,mode=LIGHT_MODE,tile=128,progress=gr.Progress()):
    image = open_image(value)
    if factor not in (1.5,2,4): raise gr.Error('Chọn 1.5×, 2× hoặc 4×.')
    if mode not in (LIGHT_MODE,AI_MODE): raise gr.Error('Chế độ upscale không hợp lệ.')
    size = (round(image.width*factor),round(image.height*factor))
    if size[0]*size[1] > MAX_IMAGE_PIXELS: raise gr.Error('Ảnh kết quả vượt 16 megapixel. Chọn hệ số nhỏ hơn.')
    started = time.monotonic()
    if mode == AI_MODE:
        if tile not in (64,128,256): raise gr.Error('Tile phải là 64, 128 hoặc 256.')
        # Anime6B luôn chạy x4 trước, sau đó resize về hệ số yêu cầu.
        if image.width*image.height > 4_000_000:
            raise gr.Error('AI giới hạn ảnh đầu vào 4 megapixel vì có ảnh trung gian x4. Hãy dùng ảnh nhỏ hơn.')
        result = run_ai_upscale(image,tile,progress)
        if result.size != size: result = result.resize(size,Image.Resampling.LANCZOS)
        method = AI_MODE
    else:
        result = image.resize(size,Image.Resampling.LANCZOS)
        method = 'Lanczos (không phải AI)'
    if sharpen:
        result = result.filter(ImageFilter.UnsharpMask(radius=1.2,percent=110,threshold=3))
    data = dict(mode=method,factor=factor,sharpen=bool(sharpen),original_size=image.size,output_size=size)
    if mode == AI_MODE:
        data.update(model='RealESRGAN_x4plus_anime_6B',native_scale=4,tile=int(tile),tile_padding=64,
                    weights_url=AI_MODEL_URL)
    png,meta = save_edit(result,'upscale',data)
    return png,png,meta,f'{image.width}×{image.height} → {size[0]}×{size[1]} • {method} • {time.monotonic()-started:.1f}s. AI có thể thay đổi chi tiết; không sửa lỗi giải phẫu.'

if "demo" in globals(): demo.close()
with gr.Blocks(title="Animagine • Anime Studio", theme=gr.themes.Soft(), delete_cache=(3600,86400)) as demo:
    gr.Markdown("# ✨ Anime Studio\nAnimagine XL 4.0 · LPW · Euler a — tạo ảnh với prompt dài hơn 100 token")
    with gr.Tab("Tạo ảnh"):
        with gr.Row():
            with gr.Column():
                pos = gr.Textbox(value=prompt.strip(), label="Prompt", lines=9)
                neg = gr.Textbox(value=negative_prompt.strip(), label="Negative prompt", lines=3)
                with gr.Accordion("Tìm và thêm tag từ CSV", open=False):
                    gr.Markdown("CSV Danbooru/e621: có thể chứa thẻ người lớn. Không phải mọi thẻ đều được model hiểu. Loại thẻ hiển thị bằng ID gốc; không đoán nghĩa các ID của bản merged.")
                    upload_tags = gr.File(label="CSV tùy chọn (bỏ trống để tải từ GitHub)", file_types=[".csv"], type="filepath")
                    load_button = gr.Button("Nạp CSV")
                    tag_status = gr.Textbox(label="Danh sách thẻ", interactive=False)
                    category = gr.Dropdown(["Tất cả"], value="Tất cả", label="Category ID")
                    query = gr.Textbox(label="Tìm tên hoặc alias (ít nhất 2 ký tự)", placeholder="Ví dụ: silver hair")
                    search_button = gr.Button("Tìm tag")
                    selected_tags = gr.Dropdown(choices=[], multiselect=True, label="Chọn tag (tối đa 80 kết quả tìm kiếm)")
                    spaces = gr.Checkbox(value=True, label="Đổi dấu _ thành khoảng trắng khi chèn")
                    with gr.Row():
                        to_positive = gr.Button("Thêm vào prompt")
                        to_negative = gr.Button("Thêm vào negative")
                check = gr.Button("Kiểm tra token")
                report = gr.Textbox(label="Token", interactive=False)
                size = gr.Dropdown(list(SIZES), value=next(iter(SIZES)), label="Kích thước")
                with gr.Row():
                    steps = gr.Slider(10,40,value=28,step=1,label="Steps")
                    cfg = gr.Slider(1,10,value=5,step=0.5,label="CFG")
                seed = gr.Number(value=-1,precision=0,label="Seed (-1 = ngẫu nhiên)")
                run = gr.Button("Tạo ảnh", variant="primary")
            with gr.Column():
                preview = gr.Image(label="Ảnh kết quả",type="filepath",interactive=False)
                status = gr.Textbox(label="Trạng thái / seed",interactive=False)
                png = gr.File(label="Tải PNG",interactive=False)
                meta = gr.File(label="Tải thông số JSON",interactive=False)
                with gr.Row():
                    to_edit = gr.Button("Chuyển sang Sửa vùng")
                    to_up = gr.Button("Chuyển sang Phóng to")
    with gr.Tab("Sửa vùng"):
        gr.Markdown("### Tô vùng cần sửa\nTô cả khớp/cổ tay để model có ngữ cảnh. Mắt thường nên sửa vùng mặt đủ rộng. Model Animagine không chuyên inpainting nên có thể cần thử nhiều seed.")
        before_state = gr.State(None)
        with gr.Row():
            with gr.Column():
                editor = gr.ImageEditor(type="pil", image_mode="RGBA", label="Ảnh gốc + mask (dùng cọ tô)",
                    brush=gr.Brush(colors=["#ffffff"],color_mode="fixed"),transforms=(),height=500)
                edit_p = gr.Textbox(label="Prompt vùng sửa",value="hand holding an umbrella, natural grip, five fingers, clean line art, masterpiece, high score",lines=3)
                edit_n = gr.Textbox(label="Negative",value=negative_prompt.strip(),lines=2)
                strength = gr.Slider(0.15,1,value=0.55,step=0.05,label="Độ mạnh sửa (cao = thay đổi nhiều)")
                with gr.Row():
                    edit_steps = gr.Slider(10,40,value=28,step=1,label="Steps")
                    edit_cfg = gr.Slider(1,10,value=5,step=0.5,label="CFG")
                edit_seed = gr.Number(value=-1,precision=0,label="Seed (-1 ngẫu nhiên)")
                crop = gr.Checkbox(value=True,label="Crop và phóng vùng sửa để model thấy rõ")
                padding = gr.Slider(0,256,value=96,step=8,label="Ngữ cảnh quanh mask (pixel)")
                feather = gr.Slider(0,32,value=8,step=1,label="Làm mềm mép mask (pixel)")
                edit_run = gr.Button("Sửa vùng đã tô",variant="primary")
            with gr.Column():
                original_view = gr.Image(label="Trước sửa",interactive=False,type="pil")
                edited = gr.Image(label="Sau sửa",interactive=False,type="filepath")
                edit_status = gr.Textbox(label="Kết quả",interactive=False)
                edit_png = gr.File(label="PNG đã sửa",interactive=False)
                edit_meta = gr.File(label="Thông số sửa JSON",interactive=False)
                with gr.Row():
                    continue_edit = gr.Button("Dùng kết quả để sửa tiếp")
                    undo = gr.Button("Hoàn tác 1 bước")
                edit_to_up = gr.Button("Chuyển ảnh đã sửa sang Phóng to")
    with gr.Tab("Phóng to"):
        gr.Markdown("### Phóng to anime 1.5× / 2× / 4×\nChọn **Real-ESRGAN Anime6B (AI)** để tăng chi tiết nét vẽ, hoặc **Lanczos** để phóng nhẹ trên CPU. AI tải model ~18 MB lần đầu, luôn chạy x4 rồi resize về hệ số yêu cầu. Có thể thay đổi chi tiết/texture; không sửa tay chân sai. Nên sửa vùng trước, upscale sau. Giới hạn ảnh AI đầu vào 4 MP, đầu ra 16 MP.")
        with gr.Row():
            with gr.Column():
                up_input = gr.Image(type="pil",label="Ảnh cần phóng to")
                mode = gr.Radio([AI_MODE,LIGHT_MODE],value=AI_MODE,label="Phương pháp")
                factor = gr.Radio([1.5,2,4],value=2,label="Hệ số")
                tile = gr.Dropdown([64,128,256],value=128,label="Tile AI (64 ít VRAM hơn, chậm hơn)")
                sharpen = gr.Checkbox(value=False,label="Làm nét nhẹ (có thể gây viền halo)")
                up_run = gr.Button("Phóng to",variant="primary")
            with gr.Column():
                up_result = gr.Image(type="filepath",label="Ảnh phóng to",interactive=False)
                up_status = gr.Textbox(label="Thông tin",interactive=False)
                up_png = gr.File(label="Tải PNG phóng to",interactive=False)
                up_meta = gr.File(label="Thông số JSON",interactive=False)
    to_edit.click(editor_value,preview,editor,api_name=False)
    to_up.click(open_image,preview,up_input,api_name=False)
    edit_run.click(repair,[editor,edit_p,edit_n,strength,edit_steps,edit_cfg,edit_seed,crop,padding,feather],
        [original_view,edited,edit_png,edit_meta,before_state,edit_status],
        concurrency_id="gpu",concurrency_limit=1,api_name=False)
    continue_edit.click(use_repaired,edited,editor,api_name=False)
    undo.click(undo_repair,before_state,[editor,original_view,edited,edit_png,edit_meta,before_state,edit_status],api_name=False)
    edit_to_up.click(open_image,edited,up_input,api_name=False)
    up_run.click(upscale,[up_input,factor,sharpen,mode,tile],[up_result,up_png,up_meta,up_status],
                 concurrency_id="gpu",concurrency_limit=1,api_name=False)
    gr.Markdown("Lưu ảnh về máy trước khi Colab dừng. Chỉ chia sẻ link với người bạn tin tưởng.")
    load_button.click(load_tags, upload_tags, [tag_status,category], api_name=False)
    search_button.click(search_tags, [query,category], selected_tags, api_name=False)
    query.submit(search_tags, [query,category], selected_tags, api_name=False)
    to_positive.click(append_tags, [pos,selected_tags,spaces], pos, api_name=False).then(token_report,[pos,neg],report,api_name=False)
    to_negative.click(append_tags, [neg,selected_tags,spaces], neg, api_name=False).then(token_report,[pos,neg],report,api_name=False)
    check.click(token_report,[pos,neg],report,api_name=False)
    run.click(generate,[pos,neg,size,steps,cfg,seed],[preview,png,meta,status],
              concurrency_id="gpu",concurrency_limit=1,api_name=False)
demo.queue(max_size=4,default_concurrency_limit=1)


## 5. Mở link gradio.live
Đặt mật khẩu khi được hỏi (không hiển thị/không lưu trong notebook). Tên đăng nhập là **anime**. Ô này in link chia sẻ; mở link và đăng nhập. Không chia sẻ output notebook có thông tin riêng tư.

In [ ]:
from getpass import getpass
password = getpass("Đặt mật khẩu giao diện (ít nhất 8 ký tự): ")
if len(password) < 8:
    raise ValueError("Mật khẩu cần ít nhất 8 ký tự. Chạy lại ô này.")
try:
    demo.launch(share=True, server_name="0.0.0.0", auth=("anime",password),
                show_error=False, inline=False)
finally:
    del password


## Khắc phục lỗi / ghi chú
- Sửa vùng: chạy model hiện có, không tải model riêng. Kết quả chưa được thử trên GPU Colab. Ngoài mask đã làm mềm giữ nguyên pixel gốc; feather lớn có thể ảnh hưởng vùng lân cận.
- Hoàn tác chỉ bước sửa gần nhất. Khi đổi ảnh trong editor, trước khi sửa mới hãy kiểm tra đúng ảnh và xóa mask cũ.
- Phóng to: chọn Real-ESRGAN Anime6B hoặc Lanczos. AI có thể tái diễn giải chi tiết, cần so sánh với ảnh gốc; sửa lỗi trước khi phóng to.
- Không tải được model AI: kiểm tra Internet, chạy lại hoặc dùng Lanczos. Hết VRAM: giảm tile xuống 64. Kết quả AI GPU và chuyển qua lại với diffusion chưa được kiểm thử tại đây.
- Mở lại ô 5 khi tunnel hết hạn; runtime dừng thì phải tải lại model. Không có link cố định.
- Không cấp GPU: kiểm tra loại runtime/quota. Không thử lách chính sách Colab.
- Import lỗi: dùng runtime mới, cài ô 1, restart rồi chạy ô 2 trở đi.
- Hết bộ nhớ: giảm độ phân giải và số token; giảm steps chủ yếu giảm thời gian, không giảm đáng kể đỉnh VRAM.
- Đã bỏ `max_embeddings_multiples` của notebook cũ: pipeline **SDXL LPW này không dùng tham số đó**; tự chia prompt thành đoạn. Không dùng nó để giới hạn bộ nhớ.
- Notebook chưa được kiểm thử sinh ảnh trên GPU Colab trong môi trường phát triển này.

Nguồn: [Model](https://huggingface.co/cagliostrolab/animagine-xl-4.0), [LPW v0.35.1](https://github.com/huggingface/diffusers/blob/v0.35.1/examples/community/lpw_stable_diffusion_xl.py), [Gradio sharing](https://www.gradio.app/guides/sharing-your-app), [Colab FAQ](https://research.google.com/colaboratory/faq.html).


## Nguồn và giấy phép kiến trúc upscale
Kiến trúc RRDB x4 chuyển thể từ [BasicSR](https://github.com/XPixelGroup/BasicSR/blob/master/basicsr/archs/rrdbnet_arch.py), Copyright 2018–2022 BasicSR Authors, Apache-2.0. Thay đổi: chỉ x4, 6 block, bỏ registry và khởi tạo phục vụ training, không phụ thuộc torchvision. Model weights lấy từ [Real-ESRGAN release v0.2.2.4](https://github.com/xinntao/Real-ESRGAN/releases/tag/v0.2.2.4).

```text
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright 2018-2022 BasicSR Authors

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

```
